# Домашнее задание к семинарам 08-09 (HW08-09)

Тема: PyTorch 101 b основы оптимизации обучения.
Часть S08: MLP b регуляризация (Dropout, BatchNorm, EarlyStopping).
Часть S09: learning rate диагностика, Adam vs SGD+momentum, weight decay.

---

## 1. Цель

Закрепить:

- базовые сущности PyTorch: `Tensor`, `Dataset/DataLoader`, `nn.Module`, loss, optimizer, цикл обучения;
- корректный режим обучения/валидации: `model.train()` / `model.eval()` и `torch.no_grad()`;
- практическое управление переобучением: **Dropout**, **BatchNorm**, **EarlyStopping**;
- базовые рычаги обучения: learning rate (плохой/хороший), сравнение Adam и SGD+momentum, использование weight decay;
- аккуратное оформление результатов: один ноутбук, короткий отчёт, артефакты эксперимента.

---

## 2. Задание

### 2.1. Структура для HW08-09 (обязательно)

1. В корне репозитория должна быть папка `homeworks/` (создать, если её ещё нет).
2. Внутри `homeworks/` создать папку `HW08-09/`.
3. В папке `homeworks/HW08-09/` создать:

- основной ноутбук: `HW08-09.ipynb`
- отчёт: `report.md`
- папку для артефактов: `artifacts/`
  - рекомендуется внутри `artifacts/` завести подпапку `figures/` для графиков

> Имена папок и файлов должны быть строго такими, как указано (регистр важен).

---

### 2.2. Учебные датасеты (выбрать один из трёх)

Выберите **один** датасет из списка (все доступны через `torchvision`):

- Вариант B: EMNIST(split="balanced")

Требования к данным:

- используйте стандартный train/test из torchvision;
- сделайте **валидацию**: отделите `val` от `train` (например, 80/20), разбиение должно быть воспроизводимым (фиксированный seed);
- путь к данным в ноутбуке – стандартный (через torchvision), без абсолютных путей к домашним каталогам.

---

### 2.3. Содержание ноутбука `HW08-09.ipynb` (обязательно)

В ноутбуке `homeworks/HW08-09/HW08-09.ipynb` реализуйте и покажите следующие блоки.

#### 2.3.1. Импорты, seed и устройство

1) Импортировать библиотеки: `torch`, `torchvision`, `numpy`, `matplotlib` (и всё, что нужно по делу).  
2) Зафиксировать seed (минимум `torch.manual_seed`, желательно также `numpy`).  
3) Определить устройство (`cuda` при наличии, иначе `cpu`) и убедиться, что и модель, и батчи переводятся на один device.

#### 2.3.2. Данные и DataLoader

1) Загрузить выбранный датасет через `torchvision.datasets.*`.  
2) Определить `transform` (минимум `ToTensor()`, нормализация – по желанию).  
3) Сделать разбиение `train/val` из train-части с фиксированным seed.  
4) Создать `DataLoader` для train/val/test.  
5) Показать sanity-check: размеры батча, shapes (`x.shape`, `y.shape`), диапазоны значений.

#### 2.3.3. Модель MLP и цикл обучения

1) Реализовать MLP как `nn.Module` (Flatten, Linear, …, logits).  
2) Выбрать:

    - loss: `CrossEntropyLoss`;
    - optimizer: `Adam` (по умолчанию для базовых экспериментов);
    - метрика: accuracy (достаточно).  

3) Реализовать функции (или эквивалент):

    - `train_one_epoch(...)`
    - `evaluate(...)` (с `model.eval()` и `torch.no_grad()`)  

4) Логировать историю обучения: train/val loss и train/val accuracy по эпохам.

---

In [1]:
# 1) Импорты
import random
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms

import matplotlib.pyplot as plt


# 2) Повторяемость
SEED = 42
random.seed(SEED)          # случайности Python
np.random.seed(SEED)       # случайности NumPy
torch.manual_seed(SEED)    # случайности PyTorch (CPU)
torch.cuda.manual_seed_all(SEED)  # случайности PyTorch (GPU)

# (для полной воспроизводимости на GPU)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 3) Устройство: GPU если доступен, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device:', device)
print('torch:', torch.__version__)
print("torchvision:", torchvision.__version__)

device: cpu
torch: 2.10.0+cpu
torchvision: 0.25.0+cpu


In [2]:
#2.3.2 Данные и DataLoader (KMNIST)

DATA_DIR = "./data" 

#2) Transform (ToTensor + нормализация)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),         
])

#1) Загрузить выбранный датасет через torchvision.datasets

# Важно: для EMNIST нужно указать split="balanced"
train_full = datasets.EMNIST(
    root=DATA_DIR, split="balanced", train=True, download=True, transform=transform
)
test_ds = datasets.EMNIST(
    root=DATA_DIR, split="balanced", train=False, download=True, transform=transform
)

print("Train full size:", len(train_full))
print("Test size:", len(test_ds))
print("Num classes:", len(train_full.classes))  # пригодится для num_classes в MLP

#3) Сделаем разбиение train/val из train-части с фиксированным seed

val_ratio = 0.2
val_size = int(len(train_full) * val_ratio)
train_size = len(train_full) - val_size

gen = torch.Generator().manual_seed(SEED)  
train_ds, val_ds = random_split(train_full, [train_size, val_size], generator=gen)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

# 4) Создать DataLoader для train/val/test

BATCH_SIZE = 256 if device.type == "cuda" else 64
NUM_WORKERS = 0 if os.name == "nt" else 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

#5) Sanity-check (батч, shapes, диапазоны)

x, y = next(iter(train_loader))

print("Batch size:", x.shape[0])
print("x.shape:", x.shape, "| y.shape:", y.shape)
print("x dtype:", x.dtype, "| y dtype:", y.dtype)
print("x min/max:", x.min().item(), x.max().item())
print("y min/max:", y.min().item(), y.max().item())


Train full size: 112800
Test size: 18800
Num classes: 47
Train size: 90240
Val size: 22560
Batch size: 64
x.shape: torch.Size([64, 1, 28, 28]) | y.shape: torch.Size([64])
x dtype: torch.float32 | y dtype: torch.int64
x min/max: -1.0 1.0
y min/max: 0 46


In [3]:
#2.3.3 Модель MLP и цикл обучения

num_classes = len(train_full.classes)  # EMNIST balanced

#1) Реализовать MLP как nn.Module (Flatten, Linear, …, logits)

class MLP(nn.Module):
    def __init__(self, hidden_sizes=(512, 256), num_classes=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                         # (B,1,28,28) -> (B,784)
            nn.Linear(28 * 28, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], num_classes)  # logits
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)

#2) Выбрать: Loss, optimizer, accuracy

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy_from_logits(logits, y_true):
    preds = logits.argmax(dim=1)
    return (preds == y_true).float().mean().item()

#3) Реализовать функции (или эквивалент)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc  += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc  += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

#4) Логировать историю обучения: train/val loss и train/val accuracy по эпохам

num_epochs = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{num_epochs} | "
        f"train loss={train_loss:.4f}, acc={train_acc:.4f} | "
        f"val loss={val_loss:.4f}, acc={val_acc:.4f}"
    )

MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Linear(in_features=256, out_features=47, bias=True)
  )
)
Epoch 01/10 | train loss=1.0390, acc=0.6876 | val loss=0.7191, acc=0.7671
Epoch 02/10 | train loss=0.6053, acc=0.7991 | val loss=0.5963, acc=0.8075
Epoch 03/10 | train loss=0.5142, acc=0.8257 | val loss=0.5338, acc=0.8232
Epoch 04/10 | train loss=0.4663, acc=0.8381 | val loss=0.5359, acc=0.8183
Epoch 05/10 | train loss=0.4278, acc=0.8482 | val loss=0.5173, acc=0.8296
Epoch 06/10 | train loss=0.4001, acc=0.8564 | val loss=0.5156, acc=0.8311
Epoch 07/10 | train loss=0.3789, acc=0.8610 | val loss=0.5214, acc=0.8310
Epoch 08/10 | train loss=0.3549, acc=0.8688 | val loss=0.5154, acc=0.8364
Epoch 09/10 | train loss=0.3368, acc=0.8727 | val loss=0.5358, acc=0.8363
Epoch 10/10 | train loss=0.3200, acc=0.8780

## 3. Эксперименты

В домашней работе есть две части: (A) регуляризация (S08) и (B) оптимизация (S09).
Все результаты прогонов фиксируются в `artifacts/runs.csv` (см. раздел 4).

### 3.1. Часть A (S08): регуляризация и переобучение (обязательно)
Задача: показать эффект Dropout/BatchNorm и ранней остановки.

Провести 4 эксперимента (E1-E4). Эксперимент = фиксированный конфиг модели/обучения + обучение + оценка на val.

- **E1 (base)**: MLP побольше (например, 2-3 скрытых слоя), без Dropout и без BatchNorm.
- **E2 (Dropout)**: как E1, но добавить Dropout (например, p=0.2-0.5).
- **E3 (BatchNorm)**: как E1, но добавить BatchNorm (между Linear и активацией).
- **E4 (EarlyStopping)**: выбрать **лучший** из (E2/E3) по `val_accuracy` и обучить его с EarlyStopping (patience 3-5).
  - именно E4 считается "лучшей моделью домашки" и из него сохраняется `best_model.pt`.

Важно:

- сравнение должно быть корректным: одинаковые split/train settings (где уместно), одинаковое число эпох (кроме early stopping);
- test-часть используйте **один раз**: только для финальной оценки лучшей модели после выбора по val.

In [4]:
# Подготовка: три модели 

# E1 base: 3 скрытых слоя, без Dropout/BN

hidden_sizes_big = (512, 256, 128)   # "MLP побольше"
lr = 1e-3
num_epochs_exp = 10                  # одинаково для E1-E3

class MLP_Base(nn.Module):
    def __init__(self, hidden_sizes=hidden_sizes_big, num_classes=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, hidden_sizes[0]), nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]), nn.ReLU(),
            nn.Linear(hidden_sizes[1], hidden_sizes[2]), nn.ReLU(),
            nn.Linear(hidden_sizes[2], num_classes)  # logits
        )

    def forward(self, x):
        return self.net(x)

# E2 Dropout: как E1, но Dropout p=0.3

p_dropout = 0.3

class MLP_Dropout(nn.Module):
    def __init__(self, hidden_sizes=hidden_sizes_big, p=p_dropout, num_classes=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, hidden_sizes[0]), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden_sizes[1], hidden_sizes[2]), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden_sizes[2], num_classes)  # logits
        )

    def forward(self, x):
        return self.net(x)

# E3 BatchNorm: как E1, но BN между Linear и ReLU

class MLP_BN(nn.Module):
    def __init__(self, hidden_sizes=hidden_sizes_big, num_classes=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),

            nn.Linear(28*28, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),

            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),

            nn.Linear(hidden_sizes[1], hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),

            nn.Linear(hidden_sizes[2], num_classes)  # logits
        )

    def forward(self, x):
        return self.net(x)


In [5]:
# E1: обучаем base 

torch.manual_seed(SEED)

model_E1 = MLP_Base().to(device)
optimizer_E1 = torch.optim.Adam(model_E1.parameters(), lr=lr)

history_E1 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, num_epochs_exp + 1):
    train_loss, train_acc = train_one_epoch(model_E1, train_loader, criterion, optimizer_E1, device)
    val_loss, val_acc     = evaluate(model_E1, val_loader, criterion, device)

    history_E1["train_loss"].append(train_loss)
    history_E1["train_acc"].append(train_acc)
    history_E1["val_loss"].append(val_loss)
    history_E1["val_acc"].append(val_acc)

    print(f"E1 | epoch {epoch:02d}/{num_epochs_exp} | tr acc={train_acc:.4f} | va acc={val_acc:.4f}")

best_val_acc_E1 = max(history_E1["val_acc"])
print("E1 best val acc:", best_val_acc_E1)

E1 | epoch 01/10 | tr acc=0.6701 | va acc=0.7732
E1 | epoch 02/10 | tr acc=0.7948 | va acc=0.7938
E1 | epoch 03/10 | tr acc=0.8201 | va acc=0.8156
E1 | epoch 04/10 | tr acc=0.8368 | va acc=0.8268
E1 | epoch 05/10 | tr acc=0.8456 | va acc=0.8351
E1 | epoch 06/10 | tr acc=0.8541 | va acc=0.8286
E1 | epoch 07/10 | tr acc=0.8613 | va acc=0.8347
E1 | epoch 08/10 | tr acc=0.8660 | va acc=0.8360
E1 | epoch 09/10 | tr acc=0.8701 | va acc=0.8415
E1 | epoch 10/10 | tr acc=0.8741 | va acc=0.8383
E1 best val acc: 0.8415368271954674


In [6]:
# E2: обучаем Dropout

torch.manual_seed(SEED)

model_E2 = MLP_Dropout().to(device)
optimizer_E2 = torch.optim.Adam(model_E2.parameters(), lr=lr)

history_E2 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, num_epochs_exp + 1):
    train_loss, train_acc = train_one_epoch(model_E2, train_loader, criterion, optimizer_E2, device)
    val_loss, val_acc     = evaluate(model_E2, val_loader, criterion, device)

    history_E2["train_loss"].append(train_loss)
    history_E2["train_acc"].append(train_acc)
    history_E2["val_loss"].append(val_loss)
    history_E2["val_acc"].append(val_acc)

    print(f"E2 | epoch {epoch:02d}/{num_epochs_exp} | tr acc={train_acc:.4f} | va acc={val_acc:.4f}")

best_val_acc_E2 = max(history_E2["val_acc"])
print("E2 best val acc:", best_val_acc_E2)


E2 | epoch 01/10 | tr acc=0.5406 | va acc=0.7393
E2 | epoch 02/10 | tr acc=0.6890 | va acc=0.7836
E2 | epoch 03/10 | tr acc=0.7199 | va acc=0.7974
E2 | epoch 04/10 | tr acc=0.7344 | va acc=0.8023
E2 | epoch 05/10 | tr acc=0.7472 | va acc=0.8135
E2 | epoch 06/10 | tr acc=0.7522 | va acc=0.8129
E2 | epoch 07/10 | tr acc=0.7579 | va acc=0.8164
E2 | epoch 08/10 | tr acc=0.7628 | va acc=0.8226
E2 | epoch 09/10 | tr acc=0.7659 | va acc=0.8223
E2 | epoch 10/10 | tr acc=0.7700 | va acc=0.8182
E2 best val acc: 0.8226363314447592


In [7]:
# E3: обучаем BatchNorm

torch.manual_seed(SEED)

model_E3 = MLP_BN().to(device)
optimizer_E3 = torch.optim.Adam(model_E3.parameters(), lr=lr)

history_E3 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, num_epochs_exp + 1):
    train_loss, train_acc = train_one_epoch(model_E3, train_loader, criterion, optimizer_E3, device)
    val_loss, val_acc     = evaluate(model_E3, val_loader, criterion, device)

    history_E3["train_loss"].append(train_loss)
    history_E3["train_acc"].append(train_acc)
    history_E3["val_loss"].append(val_loss)
    history_E3["val_acc"].append(val_acc)

    print(f"E3 | epoch {epoch:02d}/{num_epochs_exp} | tr acc={train_acc:.4f} | va acc={val_acc:.4f}")

best_val_acc_E3 = max(history_E3["val_acc"])
print("E3 best val acc:", best_val_acc_E3)


E3 | epoch 01/10 | tr acc=0.7468 | va acc=0.8138
E3 | epoch 02/10 | tr acc=0.8250 | va acc=0.8344
E3 | epoch 03/10 | tr acc=0.8449 | va acc=0.8453
E3 | epoch 04/10 | tr acc=0.8582 | va acc=0.8518
E3 | epoch 05/10 | tr acc=0.8678 | va acc=0.8501
E3 | epoch 06/10 | tr acc=0.8750 | va acc=0.8498
E3 | epoch 07/10 | tr acc=0.8820 | va acc=0.8560
E3 | epoch 08/10 | tr acc=0.8878 | va acc=0.8536
E3 | epoch 09/10 | tr acc=0.8933 | va acc=0.8589
E3 | epoch 10/10 | tr acc=0.8967 | va acc=0.8567
E3 best val acc: 0.8589323654390935


In [8]:
# выбираем лучший из (E2/E3) по val_accuracy и обучаем с EarlyStopping

if best_val_acc_E2 >= best_val_acc_E3:
    best_kind = "dropout"
    print("Best between E2/E3: E2 (Dropout)")
else:
    best_kind = "bn"
    print("Best between E2/E3: E3 (BatchNorm)")

torch.manual_seed(SEED)

patience = 4
max_epochs_E4 = 30

# модель E4 = лучший вариант из E2/E3
if best_kind == "dropout":
    model_E4 = MLP_Dropout().to(device)
else:
    model_E4 = MLP_BN().to(device)

optimizer_E4 = torch.optim.Adam(model_E4.parameters(), lr=lr)

history_E4 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val = -1.0
bad_epochs = 0
best_state = None

for epoch in range(1, max_epochs_E4 + 1):
    train_loss, train_acc = train_one_epoch(model_E4, train_loader, criterion, optimizer_E4, device)
    val_loss, val_acc     = evaluate(model_E4, val_loader, criterion, device)

    history_E4["train_loss"].append(train_loss)
    history_E4["train_acc"].append(train_acc)
    history_E4["val_loss"].append(val_loss)
    history_E4["val_acc"].append(val_acc)

    print(f"E4 | epoch {epoch:02d}/{max_epochs_E4} | tr acc={train_acc:.4f} | va acc={val_acc:.4f}")

    if val_acc > best_val:
        best_val = val_acc
        bad_epochs = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model_E4.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"E4 EarlyStopping: stop at epoch {epoch}")
            break

# восстановить лучшие веса
model_E4.load_state_dict(best_state)
print("E4 best val acc:", best_val)

Best between E2/E3: E3 (BatchNorm)
E4 | epoch 01/30 | tr acc=0.7468 | va acc=0.8138
E4 | epoch 02/30 | tr acc=0.8250 | va acc=0.8344
E4 | epoch 03/30 | tr acc=0.8449 | va acc=0.8453
E4 | epoch 04/30 | tr acc=0.8582 | va acc=0.8518
E4 | epoch 05/30 | tr acc=0.8678 | va acc=0.8501
E4 | epoch 06/30 | tr acc=0.8750 | va acc=0.8498
E4 | epoch 07/30 | tr acc=0.8820 | va acc=0.8560
E4 | epoch 08/30 | tr acc=0.8878 | va acc=0.8536
E4 | epoch 09/30 | tr acc=0.8933 | va acc=0.8589
E4 | epoch 10/30 | tr acc=0.8967 | va acc=0.8567
E4 | epoch 11/30 | tr acc=0.9029 | va acc=0.8542
E4 | epoch 12/30 | tr acc=0.9048 | va acc=0.8570
E4 | epoch 13/30 | tr acc=0.9102 | va acc=0.8576
E4 EarlyStopping: stop at epoch 13
E4 best val acc: 0.8589323654390935


In [9]:
# Тест
test_loss, test_acc = evaluate(model_E4, test_loader, criterion, device)
print("TEST | loss:", round(test_loss, 4), "| acc:", round(test_acc, 4))

TEST | loss: 0.4507 | acc: 0.8557


### 3.2. Часть B (S09): LR, оптимизаторы, weight decay (обязательно)

Задача: научиться диагностировать плохой LR по кривым, а также руками настроить SGD+momentum и weight decay.

Делаем 3 коротких эксперимента (O1-O3) на **фиксированной архитектуре** (той же, что в E4 по слоям/нейронам/Dropout/BN).

- **O1 (LR слишком большой)**: Adam, lr = "слишком большой" (например, 1e-1). Обучить 5-8 эпох и показать, что loss/метрики ведут себя плохо.
- **O2 (LR слишком маленький)**: Adam, lr = "слишком маленький" (например, 1e-5). Обучить 5-8 эпох и показать, что обучение почти не двигается.
- **O3 (SGD+momentum + weight decay)**: SGD с momentum (например, momentum=0.9) и **weight_decay > 0** (например, 1e-4).
  - lr взять разумный (например, как в E4 или подобрать в диапазоне 1e-2…1e-3).
  - обучить 10-15 эпох (или меньше, если на CPU долго).

Важно:

- O1 и O2 нужны **только для диагностики**, их модели не сохраняются как best.
- "Лучшая модель домашки" выбирается в части A (E4) по `val_accuracy` и затем проверяется на test.

---